# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [2]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Godność osoby ludzkiej
2. Pan Wołodyjowski
3. Nie budźcie mnie
4. Szymborska. Kropki, przecinki, papierosy
5. Wspólnie. 20 lat kolektywu Sputnik Photos
6. Rodzeństwo (Ritter, Dene, Voss)
7. Męczeństwo i śmierć Marata
8. Gotyk w Karpatach
9. Fisz Emade Tworzywo w Studio
10. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk

Czas wykonania: 3.43s


In [3]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Gwałtu, co się dzieje
2. Wspólnie. 20 lat kolektywu Sputnik Photos
3. Z literaturą przez wieki
4. Nie budźcie mnie
5. Dyskusyjny Klub Po Pracy. Poezja smoleńska
6. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk
7. Rodzeństwo (Ritter, Dene, Voss)
8. Męczeństwo i śmierć Marata
9. Pan Wołodyjowski
10. Fisz Emade Tworzywo w Studio

Czas wykonania (wątki): 1.13s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [4]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [5]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.59s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [7]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

# Twój kod tutaj...

def get_cat_fact():
    return requests.get(CAT_API_URL).json().get("fact")

def sequential_fetch(n=20):
    facts = []
    start = time.time()

    for _ in range(n):
        facts.append(get_cat_fact())
    
    end = time.time()
    return facts, end - start

def threaded_fetch(n=20, workers=10):
    facts = [] 
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        results = executor.map(lambda _: get_cat_fact(), range(n))
        facts = list(results)
    
    end = time.time()
    return facts, end - start

def main():
    print("== Pobieranie sekwencyjne ==")
    _, seq_time = sequential_fetch()
    print(f"Czas sekwencyjny: {seq_time:.3f} s\n")

    print("== Pobieranie wielowątkowe ==")
    _, thr_time = threaded_fetch()
    print(f"Czas wielowątkowy: {thr_time:.3f} s\n")

    print("== Porównanie ==")
    print(f"Przyspieszenie: {seq_time / thr_time:.2f}x szybciej (wątkowo)")

if __name__ =="__main__":
    main()

== Pobieranie sekwencyjne ==
Czas sekwencyjny: 8.076 s

== Pobieranie wielowątkowe ==
Czas wielowątkowy: 2.495 s

== Porównanie ==
Przyspieszenie: 3.24x szybciej (wątkowo)


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [11]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

# Twój kod tutaj...

TOTAL_NUMBERS = 30
q = queue.Queue()

def producer():
    for i in range(1, TOTAL_NUMBERS + 1):
        q.put(i)
        print(f"[PRODUCENT] Dodano {i}")
        time.sleep(0.05)

    q.put(None)
    q.put(None)


def consumer_even():
    while True:
        item = q.get()
        if item is None:
            break
        if item % 2 == 0:
            print(f"[KONSUMENT PARZYSTE] Odebrano: {item}")
        q.task_done()

def consumer_odd():
    while True: 
        item = q.get()
        if item is None:
            break
        if item % 2 == 1:
            print(f"[KONSUMENT NIEPARZYSTE] Odebrano: {item}")
        q.task_done()

def main():
    t_prod = threading.Thread(target=producer)
    t_even = threading.Thread(target=consumer_even)
    t_odd = threading.Thread(target=consumer_odd)

    t_prod.start()
    t_even.start()
    t_odd.start()

    t_prod.join()
    t_even.join()
    t_odd.join()

    print("Zakończono przetwarzanie.")

if __name__ == "__main__":
    main()



[PRODUCENT] Dodano 1
[PRODUCENT] Dodano 2
[KONSUMENT PARZYSTE] Odebrano: 2
[PRODUCENT] Dodano 3
[KONSUMENT NIEPARZYSTE] Odebrano: 3
[PRODUCENT] Dodano 4
[KONSUMENT PARZYSTE] Odebrano: 4
[PRODUCENT] Dodano 5
[KONSUMENT NIEPARZYSTE] Odebrano: 5
[PRODUCENT] Dodano 6[KONSUMENT PARZYSTE] Odebrano: 6

[PRODUCENT] Dodano 7
[KONSUMENT NIEPARZYSTE] Odebrano: 7
[PRODUCENT] Dodano 8
[KONSUMENT PARZYSTE] Odebrano: 8
[PRODUCENT] Dodano 9
[KONSUMENT NIEPARZYSTE] Odebrano: 9
[PRODUCENT] Dodano 10
[KONSUMENT PARZYSTE] Odebrano: 10
[PRODUCENT] Dodano 11[KONSUMENT NIEPARZYSTE] Odebrano: 11

[PRODUCENT] Dodano 12
[KONSUMENT PARZYSTE] Odebrano: 12
[PRODUCENT] Dodano 13
[KONSUMENT NIEPARZYSTE] Odebrano: 13
[PRODUCENT] Dodano 14
[KONSUMENT PARZYSTE] Odebrano: 14
[PRODUCENT] Dodano 15
[KONSUMENT NIEPARZYSTE] Odebrano: 15
[PRODUCENT] Dodano 16[KONSUMENT PARZYSTE] Odebrano: 16

[PRODUCENT] Dodano 17
[KONSUMENT NIEPARZYSTE] Odebrano: 17
[PRODUCENT] Dodano 18
[KONSUMENT PARZYSTE] Odebrano: 18
[PRODUCENT] Dodano 

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [12]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

RANGE_START = 1
RANGE_END = 10_000

def sequential():
    start = time.time()

    results = [] 
    for n in range(RANGE_START, RANGE_END + 1):
        results.append(calculate_power_sum(n))

    end = time.time()
    return results, end - start

def parallel():
    start = time.time()

    with multiprocessing.Pool() as pool:
        results = pool.map(calculate_power_sum, range(RANGE_START, RANGE_END + 1))  
    
    end = time.time()
    return results, end - start

if __name__ == "__main__":
    print("== Obliczenia sekwencyjne ==")
    _, seq_time = sequential()
    print(f"Czas sekwencyjny: {seq_time:.3f} s\n")

    print("== Obliczenia równoległe (multiprocessing) ==")
    _, par_time = parallel()
    print(f"Czas równoległy: {par_time:.3f} s\n")

    print("== Porównanie ==")
    print(f"Przyspieszenie {seq_time / par_time:.2f}x szybciej (multiprocessing)")




    pass

== Obliczenia sekwencyjne ==
Czas sekwencyjny: 0.338 s

== Obliczenia równoległe ==
Czas równoległy: 0.246 s

== Porównanie ==
Przyspieszenie 1.37x szybciej (multiprocessing)
